<a href="https://colab.research.google.com/github/KaanCesur354/CENG467_TakeHome/blob/main/CENG467_TakeHomeQ3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required libraries
!pip install datasets transformers torch rouge-score nltk bert-score sumy -q
!pip install evaluate -q

import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Setup done!")
print("GPU available:", torch.cuda.is_available())

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 89.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00
Setup done!
GPU available: True


In [ ]:
from datasets import load_dataset

# Load a subset of CNN/DailyMail (full dataset is very large)
dataset = load_dataset("cnn_dailymail", "3.0.0")

# Use a small subset for speed
train_data = dataset["train"].select(range(1000))
val_data = dataset["validation"].select(range(200))
test_data = dataset["test"].select(range(200))

print(dataset)
print(f"\nUsing subset — Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")
print("\n--- Sample record ---")
print("Article (first 300 chars):", train_data[0]["article"][:300])
print("\nHighlights (summary):", train_data[0]["highlights"][:300])

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})

Using subset — Train: 1000, Val: 200, Test: 200

--- Sample record ---
Article (first 300 chars): LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappoi

Highlights (summary): Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .
Young actor says he has no plans to fritter his cash away .
Radcliffe's earnings from first five Potter films have been held in trust fund .


In [ ]:
import nltk
import numpy as np
from nltk.tokenize import sent_tokenize
from nltk.corpus import stopwords
import networkx as nx

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

stop_words = set(stopwords.words('english'))

def textrank_summarize(text, num_sentences=3):
    sentences = sent_tokenize(text)
    if len(sentences) <= num_sentences:
        return ' '.join(sentences)

    def sentence_vector(sent):
        words = [w.lower() for w in sent.split() if w.lower() not in stop_words]
        return words

    sim_matrix = np.zeros((len(sentences), len(sentences)))
    for i in range(len(sentences)):
        for j in range(len(sentences)):
            if i != j:
                words_i = set(sentence_vector(sentences[i]))
                words_j = set(sentence_vector(sentences[j]))
                if len(words_i | words_j) > 0:
                    sim_matrix[i][j] = len(words_i & words_j) / len(words_i | words_j)

    nx_graph = nx.from_numpy_array(sim_matrix)
    scores = nx.pagerank(nx_graph, max_iter=200)

    ranked = sorted(scores, key=scores.get, reverse=True)[:num_sentences]
    ranked_sorted = sorted(ranked)
    summary = ' '.join([sentences[i] for i in ranked_sorted])
    return summary

# Test on a sample
sample_article = test_data[0]["article"]
sample_highlight = test_data[0]["highlights"]

print("--- TextRank Summary ---")
print(textrank_summarize(sample_article, num_sentences=3))
print("\n--- Reference Summary ---")
print(sample_highlight)

--- TextRank Summary ---
(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. Later that month, the ICC opened a preliminary examination into the situation in Palestinian territories, paving the way for possible war crimes investigations against Israelis. "As Palestine formally becomes a State Party to the Rome Statute today, the world is also a step closer to ending a long era of impunity and injustice," he said, according to an ICC news release.

--- Reference Summary ---
Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June .
Israel and the United States opposed the move, which could open the door to war crimes investigations against Israelis .


In [ ]:
from transformers import BartForConditionalGeneration, BartTokenizer

print("Loading BART model...")
bart_tokenizer = BartTokenizer.from_pretrained("facebook/bart-large-cnn")
bart_model = BartForConditionalGeneration.from_pretrained("facebook/bart-large-cnn")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bart_model = bart_model.to(device)
print("BART ready! Device:", device)

def bart_summarize(text, max_length=130, min_length=30):
    inputs = bart_tokenizer(
        text,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    ).to(device)

    summary_ids = bart_model.generate(
        inputs["input_ids"],
        max_length=max_length,
        min_length=min_length,
        length_penalty=2.0,
        num_beams=4,
        early_stopping=True
    )
    return bart_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# Test on same sample
print("\n--- BART Summary ---")
print(bart_summarize(sample_article))
print("\n--- Reference Summary ---")
print(sample_highlight)

Loading BART model...
BART ready! Device: cuda

--- BART Summary ---
The Palestinian Authority becomes the 123rd member of the International Criminal Court. The move gives the court jurisdiction over alleged crimes in Palestinian territories. Israel and the United States opposed the Palestinians' efforts to join the body.

--- Reference Summary ---
Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June .
Israel and the United States opposed the move, which could open the door to war crimes investigations against Israelis .


In [ ]:
import evaluate

# Load metrics
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")

# Generate summaries for test set
print("Generating TextRank summaries...")
textrank_summaries = [textrank_summarize(ex["article"], num_sentences=3) for ex in test_data]

print("Generating BART summaries...")
bart_summaries = []
for i, ex in enumerate(test_data):
    summary = bart_summarize(ex["article"])
    bart_summaries.append(summary)
    if (i+1) % 50 == 0:
        print(f"  {i+1}/200 done...")

references = [ex["highlights"] for ex in test_data]
print("\nAll summaries generated!")

Generating TextRank summaries...
Generating BART summaries...
  50/200 done...
  100/200 done...
  150/200 done...
  200/200 done...

All summaries generated!


In [ ]:
print("Computing ROUGE scores...")
rouge_textrank = rouge.compute(predictions=textrank_summaries, references=references)
rouge_bart = rouge.compute(predictions=bart_summaries, references=references)

print("Computing BLEU scores...")
bleu_textrank = bleu.compute(predictions=textrank_summaries, references=[[r] for r in references])
bleu_bart = bleu.compute(predictions=bart_summaries, references=[[r] for r in references])

print("Computing METEOR scores...")
meteor_textrank = meteor.compute(predictions=textrank_summaries, references=references)
meteor_bart = meteor.compute(predictions=bart_summaries, references=references)

print("Computing BERTScore...")
bertscore_textrank = bertscore.compute(predictions=textrank_summaries, references=references, lang="en")
bertscore_bart = bertscore.compute(predictions=bart_summaries, references=references, lang="en")

print("\nAll metrics computed!")

# Summary Table
print("\n" + "="*70)
print(f"{'Metric':<20} {'TextRank':>15} {'BART':>15}")
print("="*70)
print(f"{'ROUGE-1':<20} {rouge_textrank['rouge1']:>15.4f} {rouge_bart['rouge1']:>15.4f}")
print(f"{'ROUGE-2':<20} {rouge_textrank['rouge2']:>15.4f} {rouge_bart['rouge2']:>15.4f}")
print(f"{'ROUGE-L':<20} {rouge_textrank['rougeL']:>15.4f} {rouge_bart['rougeL']:>15.4f}")
print(f"{'BLEU':<20} {bleu_textrank['bleu']:>15.4f} {bleu_bart['bleu']:>15.4f}")
print(f"{'METEOR':<20} {meteor_textrank['meteor']:>15.4f} {meteor_bart['meteor']:>15.4f}")
print(f"{'BERTScore F1':<20} {sum(bertscore_textrank['f1'])/len(bertscore_textrank['f1']):>15.4f} {sum(bertscore_bart['f1'])/len(bertscore_bart['f1']):>15.4f}")
print("="*70)

Computing ROUGE scores...
Computing BLEU scores...
Computing METEOR scores...
Computing BERTScore...

All metrics computed!

Metric                 TextRank            BART
ROUGE-1                  0.2600          0.3488
ROUGE-2                  0.0894          0.1527
ROUGE-L                  0.1807          0.2616
BLEU                     0.0575          0.1208
METEOR                   0.3006          0.3378
BERTScore F1             0.8595          0.8785


In [ ]:
print("="*80)
print("QUALITATIVE EXAMPLES")
print("="*80)

for i in range(3):
    article = test_data[i]["article"]
    reference = test_data[i]["highlights"]
    tr_summary = textrank_summaries[i]
    bart_summary = bart_summaries[i]

    print(f"\n--- Example {i+1} ---")
    print(f"ARTICLE (first 200 chars): {article[:200]}...")
    print(f"\nREFERENCE  : {reference}")
    print(f"\nTEXTRANK   : {tr_summary[:300]}...")
    print(f"\nBART       : {bart_summary}")
    print("="*80)

QUALITATIVE EXAMPLES

--- Example 1 ---
ARTICLE (first 200 chars): (CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territor...

REFERENCE  : Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June .
Israel and the United States opposed the move, which could open the door to war crimes investigations against Israelis .

TEXTRANK   : (CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. Later that month, the ICC opened a preliminary examination into the situation in Palestinian te...

BART       : The Palestinian Authority becomes the 123rd member of the International Criminal Court. The move gives the court jurisdiction over alleged crimes in 

## Discussion: Extractive vs Abstractive Summarization Trade-offs

**TextRank (Extractive):**
- Sentences are copied directly from the source — grammatically correct, no hallucination risk
- Unsupervised, fast, no GPU required
- Weakness: cannot paraphrase or synthesize; often selects longer, less focused sentences (Example 3: picks biographical trivia instead of key facts)

**BART (Abstractive):**
- Generates new text by compressing and paraphrasing the source
- Produces concise, focused summaries closer to reference highlights
- ROUGE-1: +0.089, ROUGE-2: +0.063, BLEU: +0.063, BERTScore: +0.019 over TextRank
- Pretrained on CNN/DailyMail → strong domain alignment
- Weakness: requires GPU, slower inference, occasionally omits details present in reference

**Key Insight:**
Extractive summarization preserves factual accuracy but sacrifices conciseness and synthesis.
Abstractive models like BART generate more readable and compact summaries,
but require significantly more compute and carry a risk of hallucination.
For news summarization specifically, BART's pretraining on CNN/DailyMail gives it
a strong advantage over unsupervised extractive methods.